# eda

In [2]:
import pandas as pd
import numpy as np

# 1. تحميل بيانات التدريب والاختبار
train_df = pd.read_csv('train-test.csv') # تأكد من مسار المجلد الصحيح حسب مكان حفظ الملفات

print("--- معلومات عامة عن البيانات ---")
print(train_df.info())

print("\n--- أول 5 أسطر من البيانات ---")
display(train_df.head())

print("\n--- القيم المفقودة (Missing Values) ---")
print(train_df.isnull().sum()[train_df.isnull().sum() > 0])

print("\n--- الإحصائيات الوصفية للأعمدة الرقمية ---")
print(train_df.describe())

--- معلومات عامة عن البيانات ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48000 entries, 0 to 47999
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   load_id       48000 non-null  object 
 1   pickup        48000 non-null  object 
 2   delivery      48000 non-null  object 
 3   pickup_lat    48000 non-null  float64
 4   pickup_lon    48000 non-null  float64
 5   delivery_lat  48000 non-null  float64
 6   delivery_lon  48000 non-null  float64
 7   distance      48000 non-null  float64
 8   equipment     48000 non-null  object 
 9   weight        47700 non-null  float64
 10  date          48000 non-null  object 
 11  market_index  47626 non-null  float64
 12  quote_signal  48000 non-null  float64
 13  posted_rate   48000 non-null  float64
dtypes: float64(9), object(5)
memory usage: 5.1+ MB
None

--- أول 5 أسطر من البيانات ---


,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,0.98480,2.56300,1380.28



--- القيم المفقودة (Missing Values) ---
weight          300
market_index    374
dtype: int64

--- الإحصائيات الوصفية للأعمدة الرقمية ---
         pickup_lat    pickup_lon  delivery_lat  delivery_lon      distance  \
count  48000.000000  48000.000000  48000.000000  48000.000000  48000.000000   
mean      35.647545    -90.928964     35.641175    -90.857310   1135.856654   
std        4.315285     13.482431      4.317199     13.476589    728.564416   
min       28.357650   -121.698490     28.357650   -121.698490     70.000000   
25%       31.986910    -98.400590     31.986910    -98.400590    550.400000   
50%       35.294790    -88.089150     35.294790    -87.528710    953.300000   
75%       39.411040    -83.285060     39.411040    -83.285060   1645.525000   
max       44.302960    -69.500000     44.302960    -69.500000   3439.800000   

             weight  market_index  quote_signal   posted_rate  
count  47700.000000  47626.000000  48000.000000  48000.000000  
mean   31028.844004   

# preprocessing

In [3]:
import pandas as pd
import numpy as np

def clean_and_engineer_features(df):
    # 1. معالجة الوزن السالب (تحويل القيم السالبة إلى موجبة أو NaN)
    # الأرجح أنها أخطاء إدخال ويمكن أخذ القيمة المطلقة أو معاملتها كقيم مفقودة
    df['weight'] = df['weight'].abs()
    
    # 2. تعبئة القيم المفقودة (Imputation)
    df['weight'].fillna(df['weight'].median(), inplace=True)
    df['market_index'].fillna(df['market_index'].median(), inplace=True)
    
    # 3. هندسة الميزات الزمنية من عمود date
    df['date'] = pd.to_datetime(df['date'])
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['dayofweek'] = df['date'].dt.dayofweek
    
    return df

# تطبيق الدالة على بيانات التدريب
train_df = pd.read_csv('train-test.csv')
train_df = clean_and_engineer_features(train_df)

print("تم تنظيف البيانات بنجاح وجاهزة لمرحلة الترميز والنمذجة!")
print(train_df[['weight', 'market_index', 'month', 'dayofweek']].head())

تم تنظيف البيانات بنجاح وجاهزة لمرحلة الترميز والنمذجة!
    weight  market_index  month  dayofweek
0  30658.0       0.95684      1          2
1  17555.0       0.97623      1          2
2  31721.0       1.00971      1          2
3  32333.0       0.94518      1          2
4  35183.0       0.98480      1          2


# train

In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

# 1. تحميل البيانات وتطبيق التنظيف والهندسة الزمنية
train_df = pd.read_csv('train-test.csv')

# تنظيف الوزن والتعامل مع القيم المفقودة
train_df['weight'] = train_df['weight'].abs()
train_df['weight'].fillna(train_df['weight'].median(), inplace=True)
train_df['market_index'].fillna(train_df['market_index'].median(), inplace=True)

# استخراج ميزات التاريخ
train_df['date'] = pd.to_datetime(train_df['date'])
train_df['month'] = train_df['date'].dt.month
train_df['day'] = train_df['date'].dt.day
train_df['dayofweek'] = train_df['date'].dt.dayofweek

# ترميز عمود equipment باستخدام LabelEncoder
le = LabelEncoder()
train_df['equipment_encoded'] = le.fit_transform(train_df['equipment'])

# تحديد الميزات والمستهدف
features = [
    'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 
    'distance', 'weight', 'market_index', 'quote_signal', 
    'month', 'day', 'dayofweek', 'equipment_encoded'
]
target = 'posted_rate'

X = train_df[features]
y = train_df[target]

# 2. تقسيم بيانات التدريب والتحقق
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. بناء وتدريب نموذج XGBoost Regressor
model = xgb.XGBRegressor(
    n_estimators=500, 
    learning_rate=0.05, 
    max_depth=6, 
    random_state=42,
    early_stopping_rounds=50
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)

# 4. تقييم الأداء على مجموعة التحقق
preds = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, preds))
mae = mean_absolute_error(y_val, preds)

print(f"\nنتائج التقييم محلياً:")
print(f"Validation RMSE: {rmse:.4f}")
print(f"Validation MAE: {mae:.4f}")

[0]	validation_0-rmse:1399.84566
[50]	validation_0-rmse:541.79620
[100]	validation_0-rmse:534.43470
[129]	validation_0-rmse:537.09455

نتائج التقييم محلياً:
Validation RMSE: 533.2055
Validation MAE: 115.9680


# predict

In [38]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

print("1. تجهيز وتدريب النموذج على كامل بيانات التدريب...")
train_df = pd.read_csv('train-test.csv')

# تنظيف البيانات
train_df['weight'] = train_df['weight'].abs()
train_df['weight'].fillna(train_df['weight'].median(), inplace=True)
train_df['market_index'].fillna(train_df['market_index'].median(), inplace=True)

# استخراج الميزات الزمنية
train_df['date'] = pd.to_datetime(train_df['date'])
train_df['month'] = train_df['date'].dt.month
train_df['day'] = train_df['date'].dt.day
train_df['dayofweek'] = train_df['date'].dt.dayofweek

# ترميز عمود equipment
le = LabelEncoder()
train_df['equipment_encoded'] = le.fit_transform(train_df['equipment'])

features = [
    'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 
    'distance', 'weight', 'market_index', 'quote_signal', 
    'month', 'day', 'dayofweek', 'equipment_encoded'
]
target = 'posted_rate'

X = train_df[features]
y = train_df[target]

# تدريب النموذج على كامل البيانات باستخدام أفضل عدد من الأشجار (مثلاً 130 بناءً على النتائج السابقة)
model = xgb.XGBRegressor(
    n_estimators=130, 
    learning_rate=0.05, 
    max_depth=6, 
    random_state=42
)
model.fit(X, y)

print("2. معالجة وتوقع بيانات الـ validation (12,000 حمولة)...")
val_df = pd.read_csv('validation.csv')

val_df['weight'] = val_df['weight'].abs()
val_df['weight'].fillna(train_df['weight'].median(), inplace=True)
val_df['market_index'].fillna(train_df['market_index'].median(), inplace=True)

val_df['date'] = pd.to_datetime(val_df['date'])
val_df['month'] = val_df['date'].dt.month
val_df['day'] = val_df['date'].dt.day
val_df['dayofweek'] = val_df['date'].dt.dayofweek

# معالجة التصنيفات الجديدة في حال وجدت في بيانات الـ validation
val_df['equipment_encoded'] = val_df['equipment'].apply(
    lambda x: le.transform([x])[0] if x in le.classes_ else 0
)

X_val_final = val_df[features]

# التنبؤ بالأسعار
val_preds = model.predict(X_val_final)

print("3. تعبئة القالب وحفظ الملف النهائي...")
template = pd.read_csv('validation-predictions-template.csv')
template['predicted_rate'] = val_preds

# حفظ الملف بالاسم المطلوب تماماً في التعليمات
template.to_csv('validation_predictions.csv', index=False)
print("تم بنجاح حفظ الملف باسم: validation_predictions.csv!")

1. تجهيز وتدريب النموذج على كامل بيانات التدريب...
2. معالجة وتوقع بيانات الـ validation (12,000 حمولة)...
3. تعبئة القالب وحفظ الملف النهائي...
تم بنجاح حفظ الملف باسم: validation_predictions.csv!


In [42]:
!python score.py --predictions validation_predictions.csv --december-predictions data/december-chart-inputs.csv

Validated 12,000 final predictions.
Validated 31 fixed December predictions.
Created chart: scorer_results\candidate_december.png
Final validation metrics are calculated by Spotter after submission.
